# Silkuniene et al. (2025) from dandiset 001277

Paper: Silkuniene G, Silkunas M, Pakhomova ON, Semenov I, Pakhomov AG, "MHz compression of pulse
packets facilitates remote focusing of electroporation", Bioelectrochemistry 166 (2025) 109016,
[doi:10.1016/j.bioelechem.2025.109016](https://doi.org/10.1016/j.bioelechem.2025.109016).

Dataset: [DANDI:001277](https://dandiarchive.org/dandiset/001277/0.250701.1610), version
`0.250701.1610` ([doi:10.48324/dandi.001277/0.250701.1610](https://doi.org/10.48324/dandi.001277/0.250701.1610)).
The paper cites no version (its data statement says data are available on request). This is the
only published version, and the draft has the same 49 assets and size and has not been modified
since it was published, so the most recent published version is pinned.

Claims: `claims/001277/10.1016-j.bioelechem.2025.109016.yaml`, written from the accepted author
manuscript in PubMed Central (PMC12331426), Methods, Results and legends; no supplement.

The dandiset holds one experiment of the paper: the pulse-duration comparison of Fig. 6 (CT26
cells, packets of 300, 600 or 900 ns pulses with no interval, 2 bursts (x4)). It holds 18 wells on
three plates, with a Hoechst image before exposure, a YO-PRO-1 (YP) image 30 min after it, and on
two plates a propidium (Pr) and Hoechst image 2 h after it.

| Claim | Status here |
|---|---|
| fig6b-peak-900ns-not-significant | tested |
| fig6c-normalized-decline-similar | tested |
| fig6d-longer-pulses-sharper-peak | tested |
| fig6d-600-900-differ-from-300 | tested (depends on the claim above) |
| fig6d-900ns-fwhm | tested (depends on fig6d-longer-pulses-sharper-peak) |
| fig5-yp-lorentzian-pr-gaussian | tested, on the 600-ns wells taken as the 0-mm group of Fig. 5 (assumption) |
| fig5-yp-spot-slightly-larger-than-pr | tested, as above |
| fig5-raised-electrodes-weaker-centre | blocked, data missing (no raised-electrode wells) |
| fig3-cancan-centre-enhancement, fig3-lorentzian-decline | blocked, data missing (packet-rate series and controls) |
| fig4b-suppression-near-electrodes, fig4b-centre-no-rate-effect, fig4c-fwhm-falls-with-rate | blocked, data missing (packet-rate series) |
| fig7be-burst-number-centre, fig7-fwhm-burst-number, fig7-sharpest-peak-1-burst-ct26 | blocked, data missing (burst-number series, CHO cells) |
| fig2-heating-below-44c | blocked, data missing (thermochromic film images) |

Images are streamed from the archive and reduced as they load to 40.6-um block means (the paper's
325 x 406 um ROI is exactly 8 x 10 blocks); see `cancan.py` for every parameter and its source.
Streaming takes about 8 minutes.

In [ ]:
import itertools
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import cancan as cc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
print("dandiset", cc.DANDISET, "version", cc.VERSION)

## Data check

Every file's metadata, read from the archive. The condition of each well is parsed from its
subject description.

In [ ]:
files = cc.file_table()
print(len(files), "files,", files.size_gb.sum().round(2), "GB,", files.well.nunique(), "wells")
print(files[["path", "series", "shape", "pulse_ns", "sham", "packets", "rate", "repeats", "size_gb"]].to_string())

In [ ]:
# metadata checks against the paper and the dandiset description
print("species/strain:", files.strain.unique())
print("pixel size (um):", files.pixel_um.unique(), " significant bits:", files.significant_bits.unique())
print("dtype:", files.dtype.unique())
print("series per well:")
print(files.groupby(["plate", "series"]).well.nunique().unstack(fill_value=0))
wells = (files[files.series == "Imaging_FITC"]
         [["well", "plate", "date", "plate_row", "plate_col", "pulse_ns", "sham", "packets", "rate", "repeats"]]
         .reset_index(drop=True))
print()
print("wells by plate and condition (pulse ns; sham = no pulses):")
print(pd.crosstab([wells.plate, wells.plate_row], wells.pulse_ns.fillna(0).astype(int)))
print()
print("plate layout:")
print(wells.pivot_table(index=["plate", "plate_row"], columns="plate_col", values="pulse_ns", aggfunc="first"))
print()
print("packets and packet rate stated per well, by pulse duration:")
print(wells.groupby(wells.pulse_ns.fillna(0).astype(int))[["packets", "rate", "repeats"]].agg(lambda s: sorted(set(s))))
gen = files.general_description.unique()
print()
print(len(gen), "distinct general descriptions; inter-electrode distance stated:",
      gen[0].split("inter-electrode distance of ")[1].split(" mm")[0], "mm")
print("image width:", files["shape"].iloc[0][-1] * files.pixel_um.iloc[0] / 1000, "mm")

Checks of the metadata against the paper:

- Every well is stated as "9 packets of pulses delivered at 0.2MHz", whatever its pulse duration.
  The dandiset's own description and the paper (Fig. 6A legend) give 18, 9 or 6 packets for 300,
  600 and 900 ns, all delivered with no interval (0.4, 0.2 and 0.14 MHz). The per-file text is
  wrong for the 300- and 900-ns wells; the analysis uses only the pulse duration.
- The stated inter-electrode distance, 10 mm, is the gap between the facing edges of adjacent
  electrodes. With 2.8-mm needles it gives a centre-to-centre side of 12.8 mm, a centre-to-centre
  diagonal of 18.1 mm and an edge-to-edge diagonal of 15.3 mm, the paper's figure. The two
  descriptions are consistent; the footprints found below check the geometry.
- 5 wells at 300 ns, 7 at 600 ns, 5 at 900 ns and 1 sham. The paper reports n=5 per duration
  (Fig. 6B legend). Plate P1 row B holds the sham and two 600-ns wells; the other five plate rows
  each hold one well of each duration.
- Plate P3 has no 2-h propidium image. Two files of well P2_20240925_B3 hold propidium images
  (checked below).

In [ ]:
# the two Cy3-DAPI files of P2_20240925_B3
dup = files[(files.well == "P2_20240925_B3") & (files.series == "Imaging_Cy3-DAPI")]
print(dup[["path", "size_gb", "identifier", "session_start_time", "acquisition_date"]].to_string())
digests = []
for aid in dup.asset_id:
    h = cc.open_nwb(aid)
    d = h["acquisition/Imaging_Cy3-DAPI/data"]
    band = np.stack([d[0, c, 9000:10500, :] for c in (0, 1)])
    digests.append(hashlib.md5(band.tobytes()).hexdigest())
print("md5 of rows 9000-10500, both channels:", digests, "identical:", digests[0] == digests[1])

### Loading the images

YP images of all 18 wells. Pr and the 2-h Hoechst images of the 600-ns wells that have them
(the Fig. 5 claims use only these). The duplicate P2_20240925_B3 file without an object id in its
name is not used. Channel 0 of the Cy3-DAPI files is Pr and channel 1 Hoechst: the OME
metadata of each file name them Cy3 (emission 565 nm) and DAPI (455 nm), printed below the
loading cell.

In [ ]:
fitc = files[files.series == "Imaging_FITC"].set_index("well")
cy = files[(files.series == "Imaging_Cy3-DAPI") & files.path.str.contains("_obj-")].set_index("well")
pr_wells = [w for w in cy.index if cy.loc[w, "pulse_ns"] == 600]
jobs = [(fitc.loc[w, "asset_id"], None) for w in wells.well]
jobs += [(cy.loc[w, "asset_id"], 0) for w in pr_wells] + [(cy.loc[w, "asset_id"], 1) for w in pr_wells]
loaded = cc.load_many(jobs, workers=4)
yp = {w: loaded[i][0] for i, w in enumerate(wells.well)}
k = len(wells)
pr = {w: loaded[k + i][0] for i, w in enumerate(pr_wells)}
hoechst2h = {w: loaded[k + len(pr_wells) + i][0] for i, w in enumerate(pr_wells)}
sat = {w: loaded[i][1] for i, w in enumerate(wells.well)}
top = {w: loaded[i][2] for i, w in enumerate(wells.well)}
print("YP images:", len(yp), " Pr images:", len(pr), pr_wells)
print("block image shape:", yp[wells.well[0]].shape, "block size %.4f mm" % cc.BLOCK_MM)

In [ ]:
# channel order of the Cy3-DAPI files, from the OME metadata of each file used
import re
for w in pr_wells:
    h = cc.open_nwb(cy.loc[w, "asset_id"])
    ome = h["acquisition/Imaging_Cy3-DAPI"].attrs["description"]
    ch = re.findall(r'Channel ID="Channel:(\d)" Name="([^"]+)".*?EmissionWavelength="([^"]+)"', ome)
    print(w, ["channel %s %s (emission %s nm)" % c for c in ch])
print("Cy3-DAPI files (one per well) by pulse duration:", cy.pulse_ns.fillna(0).astype(int).value_counts().sort_index().to_dict(),
      "(0 = sham)")

In [ ]:
rng_tab = pd.DataFrame({
    "YP block min": {w: float(yp[w][yp[w] > 0].min()) for w in yp},
    "YP block max": {w: float(yp[w].max()) for w in yp},
    "YP median": {w: float(np.median(yp[w][yp[w] > 0])) for w in yp},
    "max pixel": top,
    "pixels at 65535": sat,
    "unimaged blocks (zeros)": {w: float((yp[w] == 0).mean()) for w in yp},
})
print("YP intensities in camera units ('unit' attribute 'n.a.'; OME metadata says 12 significant bits):")
print(rng_tab.round(4).to_string())
for w in pr_wells:
    print(w, "Pr block range %.0f to %.0f, max pixel %d; 2-h Hoechst median %.0f"
          % (pr[w][pr[w] > 0].min(), pr[w].max(), loaded[k + pr_wells.index(w)][2], np.median(hoechst2h[w][hoechst2h[w] > 0])))
print("YP images with pixel values above 4095:", sum(v > 4095 for v in top.values()), "of", len(top))

The OME metadata of every file gives 12 significant bits, but pixel values in the YP images run
far above 4095 (the largest block means are 3500 to 22600), so the stored values are
not raw 12-bit camera counts (perhaps rescaled by the stitching software). The analysis uses
background-subtracted differences and ratios within each image, which a common scale factor does
not change. Unimaged corners of the stitched mosaic are zeros and are not used.

### Electrode footprints and the array centre

The paper measures from the geometrical centre of the array along the diagonals joining the
footprints (Methods 2.5). The footprints are found in each image (`cc.find_electrodes`). The
footprint centre-to-centre diagonal implied by Methods 2.2 is 18.1 mm (15.3 mm between facing
edges plus one 2.8-mm diameter), and the diagonals should be perpendicular. As a check that the
centre found from the footprints is where the YP response is, the brightest point of the smoothed
YP image within 5 mm of the centre is compared with it, and with 1000 random points at the same
radius from the image centre. The YP image is clipped at its 99.5th percentile and smoothed
(sigma 0.5 mm) first, so that bright debris does not count as the peak.

In [ ]:
from scipy import ndimage

meas = {}
geo_rows = []
for w in wells.well:
    sham = bool(wells.set_index("well").loc[w, "sham"])
    m = cc.measure_well(yp[w])
    meas[w] = m
    g = m["geom"]
    sm = ndimage.gaussian_filter(np.minimum(yp[w], np.percentile(yp[w], 99.5)), 12)
    c = g["centre"]
    r = int(5 / cc.BLOCK_MM)
    sub = sm[int(c[0]) - r:int(c[0]) + r, int(c[1]) - r:int(c[1]) + r]
    pk = np.array(np.unravel_index(np.argmax(sub), sub.shape)) + [int(c[0]) - r, int(c[1]) - r]
    geo_rows.append({"well": w, "pulse_ns": wells.set_index("well").loc[w, "pulse_ns"], "sham": sham,
                     "diag1_mm": g["diag_lengths_mm"][0], "diag2_mm": g["diag_lengths_mm"][1],
                     "angle_deg": g["diag_angle_deg"], "min_rim_contrast": min(g["contrast"]),
                     "peak_offset_mm": float(np.hypot(*(pk - c)) * cc.BLOCK_MM),
                     "background": m[("bg", "darkest")], "background_all4": m[("bg", "all")]})
geo = pd.DataFrame(geo_rows).set_index("well")
print(geo.round(2).to_string())
ok = ~geo.sham
print("exposed wells: diagonals %.2f to %.2f mm (mean %.2f; expected 18.1), angle %.1f to %.1f deg"
      % (geo[ok][["diag1_mm", "diag2_mm"]].min().min(), geo[ok][["diag1_mm", "diag2_mm"]].max().max(),
         geo[ok][["diag1_mm", "diag2_mm"]].values.mean(), geo[ok].angle_deg.min(), geo[ok].angle_deg.max()))
print("YP peak to array centre: median %.2f mm, max %.2f mm over %d exposed wells; %d within 1 mm"
      % (geo[ok].peak_offset_mm.median(), geo[ok].peak_offset_mm.max(), ok.sum(), (geo[ok].peak_offset_mm < 1).sum()))

In [ ]:
# reference: distance from the YP peak to random points 0 to 3 mm from the image centre
rng = np.random.default_rng(1277)
ref = []
for w in geo[ok].index:
    sm = ndimage.gaussian_filter(np.minimum(yp[w], np.percentile(yp[w], 99.5)), 12)
    c = meas[w]["geom"]["centre"]
    r = int(5 / cc.BLOCK_MM)
    sub = sm[int(c[0]) - r:int(c[0]) + r, int(c[1]) - r:int(c[1]) + r]
    pk = np.array(np.unravel_index(np.argmax(sub), sub.shape)) + [int(c[0]) - r, int(c[1]) - r]
    n = yp[w].shape[0]
    ang = rng.uniform(0, 2 * np.pi, 1000)
    rad = rng.uniform(0, 3 / cc.BLOCK_MM, 1000)
    pts = np.c_[(n - 1) / 2 + rad * np.sin(ang), (n - 1) / 2 + rad * np.cos(ang)]
    ref.append(np.median(np.hypot(*(pts - pk).T)) * cc.BLOCK_MM)
print("median distance from the YP peak to random points within 3 mm of the image centre: %.2f mm (range %.2f to %.2f)"
      % (np.median(ref), np.min(ref), np.max(ref)))

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9.5))
for ax, w in zip(axes.flat, wells.well):
    im = yp[w]
    ax.imshow(im, cmap="gray", vmin=np.percentile(im[im > 0], 1), vmax=np.percentile(im, 99.5))
    g = meas[w]["geom"]
    e = g["electrodes"]
    for a, b in ((0, 2), (1, 3)):
        ax.plot(e[[a, b], 1], e[[a, b], 0], "y-", lw=0.6)
    ax.plot(e[:, 1], e[:, 0], "r+", ms=10)
    lab = wells.set_index("well").loc[w]
    ax.set_title("%s  %s" % (w, "sham" if lab.sham else "%d ns" % lab.pulse_ns), fontsize=8)
    ax.axis("off")
fig.suptitle("YP images (1st to 99.5th percentile of each), footprints found and the diagonals")
plt.tight_layout()
plt.show()

The footprints are found in all 17 exposed wells, with diagonals of 17.7 to 18.5 mm against the
18.1 mm the paper's geometry implies, and close to perpendicular; the sham well has no footprints
(no electrodes visible) and is not used. In 16 of the 17 exposed wells the YP peak lies within
0.42 mm of the centre found from the footprints, against a median of 1.7 mm to random points
near the image centre; the exception is P1_20240926_B2, whose uptake covers the whole array. This
is consistent with the paper's geometry and the alignment of
ROIs to the array, which is the spatial counterpart of a clock check for these single-time-point
images (there are no trial tables or time series). Distances along a diagonal are scaled so
that the footprint centre is at 9.05 mm; the measured footprint diagonals, above, show the scale
of the stitched images agrees with the stated 1.625 um per pixel within 3%.

Two 600-ns wells on plate P1, row B (P1_20240926_B2 and B3), show YP uptake over the whole array,
unlike the other 15 exposed wells. Their profiles are shown next.

In [ ]:
x1, _ = meas[wells.well[0]][("darkest", "1mm")]
prof_tab = pd.DataFrame({w: meas[w][("darkest", "1mm")][1].mean(axis=0) for w in geo[ok].index}, index=x1).T
prof_tab.insert(0, "pulse_ns", geo[ok].pulse_ns.astype(int))
print("Background-subtracted YP (camera units), mean of the two diagonals, 1-mm ROI steps:")
print(prof_tab.round(0).to_string())
near = prof_tab[[-7.0, -6.0, 6.0, 7.0]].mean(axis=1)
print()
print("mean YP at 6 and 7 mm as % of the centre:")
print((100 * near / prof_tab[0.0]).round(0).to_string())

### Which wells form Fig. 6 (assumption)

The paper reports n=5 per pulse duration; the dandiset has 7 wells at 600 ns. Neither the paper
nor the dandiset says which five were used. The plate layout gives a basis that does not depend on
the results: five plate rows (P1 row A, P2 rows A and B, P3 rows A and B) each hold one well of
300, 600 and 900 ns, which is one complete experiment per row and gives exactly n=5 per duration.
The sixth row, P1 row B, holds the sham well and two more 600-ns wells. The main reading takes
the five complete rows; it is an assumption, and each Fig. 6 claim also shows the result with all
7 wells at 600 ns and with every choice of 5 of the 7. The two P1 row B wells also differ in their
images (uptake over the whole array, above) and P1_20240926_B3 shows a loss of Hoechst-stained
cells over the centre at 2 h (below), the paper's check for detachment (Methods 2.4).

In [ ]:
trip = cc.triplet_rows(wells)
print("complete rows:", sorted(trip))
wl = wells.set_index("well")
main = {d: [w for w in wl.index if wl.loc[w, "pulse_ns"] == d and (wl.loc[w, "plate"], wl.loc[w, "plate_row"]) in trip]
        for d in (300, 600, 900)}
all7 = {d: [w for w in wl.index if wl.loc[w, "pulse_ns"] == d] for d in (300, 600, 900)}
for d in (300, 600, 900):
    print(d, "ns main:", main[d], " not in main:", sorted(set(all7[d]) - set(main[d])))
six = all7[600]
subsets = [list(s) for s in itertools.combinations(six, 5)]
print(len(subsets), "subsets of 5 of the 7 600-ns wells")
# 2-h Hoechst: median of the central 6-mm disc (centre from the Pr image's footprints, the two
# channels being one acquisition) over the median of the imaged well
for w in pr_wells:
    h2 = hoechst2h[w]
    n = h2.shape[0]
    yy, xx = np.mgrid[0:n, 0:n]
    cen = cc.find_electrodes(pr[w])["centre"]
    disc = np.hypot(yy - cen[0], xx - cen[1]) * cc.BLOCK_MM < 3
    print(w, "2-h Hoechst, central 6-mm disc median / image median: %.2f" % (np.median(h2[disc]) / np.median(h2[h2 > 0])))

In [ ]:
def fw(wset, spacing, half, pooling, bg="darkest"):
    return {d: np.array([cc.fwhm(meas[w], spacing, half, pooling, bg)["fwhm"] for w in wset[d]]) for d in (300, 600, 900)}

def sets_for(sub):
    return {300: main[300], 600: sub, 900: main[900]}

readings = list(itertools.product(cc.SPACINGS, cc.HALF_RANGES, cc.POOLINGS))
fits_failed = sum(not cc.fwhm(meas[w], s, h, p)["ok"] for w in geo[ok].index for s, h, p in readings)
print("Lorentzian fits that failed:", fits_failed, "of", len(readings) * ok.sum())

## fig6b-peak-900ns-not-significant

> The central peak of YP uptake is highest for 900-ns pulse packets, but its difference from 300-
> and 600-ns packets is not statistically significant.

Per well, background-subtracted YP at the centre ROI. Readings of the criterion: (a) two-sided
t-tests of 900 against 300 and against 600 ns, Bonferroni over the three pairwise comparisons;
(b) one-way ANOVA over the three durations. The centre value from the Lorentzian fit (parameter
`a`, 1-mm steps, -5 to 5 mm) is a variant. The ROI spacing does not change the centre ROI.

In [ ]:
def fig6b(wset, bg="darkest", use_fit=False):
    if use_fit:
        c = {d: np.array([cc.fwhm(meas[w], "1mm", 5.0, "avg", bg)["a"] for w in wset[d]]) for d in (300, 600, 900)}
    else:
        c = {d: np.array([cc.centre_value(meas[w], "1mm", bg) for w in wset[d]]) for d in (300, 600, 900)}
    ms = {d: cc.mean_se(c[d]) for d in c}
    t9_3 = cc.ttest_bonf(c[900], c[300], 3)
    t9_6 = cc.ttest_bonf(c[900], c[600], 3)
    t6_3 = cc.ttest_bonf(c[600], c[300], 3)
    an = stats.f_oneway(c[300], c[600], c[900])
    top = max(ms, key=lambda d: ms[d][0])
    return {"mean_se": {d: (round(ms[d][0], 1), round(ms[d][1], 1), len(c[d])) for d in ms}, "highest": top,
            "p_bonf 900v300": t9_3["p_bonf"], "p_bonf 900v600": t9_6["p_bonf"], "p_bonf 600v300": t6_3["p_bonf"],
            "d 900v300": t9_3["cohen_d"], "d 900v600": t9_6["cohen_d"], "anova F": float(an.statistic), "anova p": float(an.pvalue)}

for name, ws, kw in [("main (5 complete rows)", main, {}), ("all 7 at 600 ns", all7, {}),
                     ("main, background from all 4 footprints", main, {"bg": "all"}),
                     ("main, centre from Lorentzian a", main, {"use_fit": True})]:
    r = fig6b(ws, **kw)
    print(name)
    print("  centre YP mean, s.e., n:", r["mean_se"], " highest:", r["highest"])
    print("  (a) Bonferroni p: 900v300 %.3f, 900v600 %.3f, 600v300 %.3f; Cohen's d 900v300 %.2f, 900v600 %.2f"
          % (r["p_bonf 900v300"], r["p_bonf 900v600"], r["p_bonf 600v300"], r["d 900v300"], r["d 900v600"]))
    print("  (b) ANOVA F = %.2f, p = %.3f" % (r["anova F"], r["anova p"]))
res = [fig6b(sets_for(s)) for s in subsets]
print("every 5 of 7 at 600 ns: 900 ns highest in %d of %d; any (a) p < 0.05 in %d; ANOVA p < 0.05 in %d"
      % (sum(r["highest"] == 900 for r in res), len(res),
         sum(min(r["p_bonf 900v300"], r["p_bonf 900v600"]) < 0.05 for r in res), sum(r["anova p"] < 0.05 for r in res)))

In [ ]:
print("centre YP per plate (main wells):")
cv = pd.DataFrame([{"well": w, "plate": wl.loc[w, "plate"], "pulse_ns": d, "centre": cc.centre_value(meas[w])}
                   for d in (300, 600, 900) for w in main[d]])
print(cv.pivot_table(index="plate", columns="pulse_ns", values="centre", aggfunc="mean").round(0))

In [ ]:
# variant: the five plate rows as blocks (each row is one experiment with one well of each duration)
rowc = pd.DataFrame({d: [cc.centre_value(meas[w]) for w in main[d]] for d in (300, 600, 900)},
                    index=[(wl.loc[w, "plate"], wl.loc[w, "plate_row"]) for w in main[300]])
assert all([(wl.loc[w, "plate"], wl.loc[w, "plate_row"]) for w in main[d]] == list(rowc.index) for d in (600, 900))
for a, b in ((900, 300), (900, 600), (600, 300)):
    t = stats.ttest_rel(rowc[a], rowc[b])
    print("paired t-test %d v %d: p = %.3f, Bonferroni (3) p = %.3f" % (a, b, t.pvalue, min(1, 3 * t.pvalue)))
y = rowc.values
gm = y.mean()
ss_d = y.shape[0] * ((y.mean(0) - gm) ** 2).sum()
ss_r = y.shape[1] * ((y.mean(1) - gm) ** 2).sum()
ss_e = ((y - gm) ** 2).sum() - ss_d - ss_r
df_d, df_e = y.shape[1] - 1, (y.shape[0] - 1) * (y.shape[1] - 1)
F = (ss_d / df_d) / (ss_e / df_e)
print("repeated-measures ANOVA (duration within row): F(%d, %d) = %.2f, p = %.3f" % (df_d, df_e, F, stats.f.sf(F, df_d, df_e)))

**Result: reproduced, with the five complete plate rows as the Fig. 6 wells (assumption).** Centre
YP is highest at 900 ns (899 against 551 at 300 ns and 866 at 600 ns, camera units), and no test
finds a difference: the Bonferroni-corrected p of 900 against 300 ns is 0.975 and against 600 ns
1.000 (Cohen's d 0.66 and 0.05), and the ANOVA p is 0.54. The same holds with all 7 wells at
600 ns, with the background from all four footprints, and with the fitted centre value. Over the
21 choices of 5 of the 7 600-ns wells no test is significant, but 900 ns is highest in only 14.
Centre YP differs more between plates than between durations (P3 two to four times P1 and P2).
The criterion's unpaired tests ignore that each plate row is one experiment; treating the rows
as blocks, paired t-tests give Bonferroni p of 0.415 (900 against 300 ns) and 1.000 (900 against
600 ns), and a repeated-measures ANOVA gives p = 0.071, so this variant also finds no difference.

## fig6c-normalized-decline-similar

> With the peak YP emission taken as 100%, the decline of YP emission away from the centre looks
> similar for 300-, 600- and 900-ns pulses.

Per well, the profile (two diagonals averaged) in percent of its value at 0 mm; mean and s.e. per
duration at each distance. Two durations differ at a distance when the gap between their s.e. bars
is longer than the bars (Methods 2.5; the mean of the two bar lengths, Cumming et al. 2007, is used
when they differ, and the longer bar is shown as a variant). Readings: distances -5 to 5 mm or every
measured distance (to 7 mm), and 1-mm steps or contiguous ROIs.

In [ ]:
def fig6c(wset, spacing, lim, bar="mean"):
    x, _ = meas[wset[300][0]][("darkest", spacing)]
    i0 = int(np.argmin(np.abs(x)))
    norm = {d: np.array([100 * meas[w][("darkest", spacing)][1].mean(0) / meas[w][("darkest", spacing)][1].mean(0)[i0]
                         for w in wset[d]]) for d in (300, 600, 900)}
    hits = []
    sel = np.where((np.abs(x) <= lim + 1e-9) & (np.arange(len(x)) != i0))[0]
    for a, b in itertools.combinations((300, 600, 900), 2):
        for i in sel:
            ma, sa = cc.mean_se(norm[a][:, i])
            mb, sb = cc.mean_se(norm[b][:, i])
            gap = max(mb - sb - (ma + sa), ma - sa - (mb + sb))
            thr = (sa + sb) / 2 if bar == "mean" else max(sa, sb)
            if gap > thr:
                hits.append((a, b, round(float(x[i]), 2), round(ma, 1), round(mb, 1), round(gap, 1), round(thr, 1)))
    return norm, x, hits

for name, ws in [("main", main), ("all 7 at 600 ns", all7)]:
    for spacing in cc.SPACINGS:
        for lim in (5.0, 7.0):
            for bar in ("mean", "longer"):
                _, _, hits = fig6c(ws, spacing, lim, bar)
                print("%-16s %-10s to %.0f mm, %-6s bar: %d distances with a gap; (a, b, x mm, mean a %%, mean b %%, gap, bar)"
                      % (name, spacing, lim, bar, len(hits)), hits)
cnt = [len(fig6c(sets_for(s), "1mm", 5.0)[2]) for s in subsets]
print("every 5 of 7 at 600 ns, 1-mm steps to 5 mm: distances with a gap per subset:", cnt)

In [ ]:
norm, x, _ = fig6c(main, "contiguous", 7.0)
fig, ax = plt.subplots(figsize=(7, 4))
for d, col in zip((300, 600, 900), ("C0", "C1", "C2")):
    m = norm[d].mean(0)
    s = norm[d].std(0, ddof=1) / np.sqrt(len(norm[d]))
    ax.errorbar(x, m, s, color=col, marker="o", ms=3, capsize=2, label="%d ns (n=%d)" % (d, len(norm[d])))
ax.set_xlabel("distance from centre along diagonals (mm)")
ax.set_ylabel("YP, % of centre")
ax.legend()
ax.set_title("Main wells, contiguous ROIs")
plt.show()
print("plotted range: %.1f to %.1f %%; nothing outside the axes" % (min(norm[d].mean(0).min() for d in norm), max(norm[d].mean(0).max() for d in norm)))

**Result: not reproduced.** Under every reading (both ROI spacings, both distance ranges, both
bar rules, the main wells or all 7 at 600 ns, and every choice of 5 of the 7) at least one
distance shows two durations separated by a gap longer than the s.e. bars. The gaps are mostly
between 300 and 900 ns at 1 to 2 mm from the centre, the 900-ns profile falling faster (19.6%
against 12.5% of the centre at +2 mm with 1-mm steps, gap 4.6 against bars of 1.3); with
contiguous ROIs 300 and 600 ns also differ at 1.6 to 4.9 mm. This is the sharper peak at longer
pulses that the paper reports in Fig. 6D; by the error-bar rule the criterion names, the declines
are not similar.

## fig6d-longer-pulses-sharper-peak

> FWHM of the central YP peak, fitted per experiment, decreases with pulse duration from 300 to
> 900 ns, with a trend slope different from zero (p = 0.003).

Per well, the Lorentzian of Methods 2.5 fitted by least squares (centre at 0, no offset); FWHM is
`b`. FWHM is regressed on pulse duration (linear, Fig. 6D legend, or on log duration, Results 3.3),
over the 15 individual values or the 3 means, with the two-sided t-test of the slope. Further
readings: fit half-range 4, 5 or 6 mm; 1-mm or contiguous ROIs; the diagonals averaged before
fitting, or all points fitted (these two give the same least-squares fit when both diagonals are
sampled at the same distances, as here).

In [ ]:
def fig6d_slopes(wset):
    rows = []
    for spacing, half, pooling in readings:
        f = fw(wset, spacing, half, pooling)
        xs = np.concatenate([[d] * len(f[d]) for d in (300, 600, 900)])
        ys = np.concatenate([f[d] for d in (300, 600, 900)])
        means = np.array([f[d].mean() for d in (300, 600, 900)])
        for points in ("individual", "means"):
            for trend in ("linear", "log"):
                xx = (np.array([300, 600, 900]) if points == "means" else xs).astype(float)
                yy = means if points == "means" else ys
                st = cc.slope_test(np.log(xx) if trend == "log" else xx, yy)
                rows.append({"spacing": spacing, "half_mm": half, "pooling": pooling, "points": points, "trend": trend,
                             "fwhm300": f[300].mean(), "fwhm600": f[600].mean(), "fwhm900": f[900].mean(),
                             "slope": st["slope"], "p": st["p"], "r": st["r"], "n": st["n"]})
    return pd.DataFrame(rows)

t_main = fig6d_slopes(main)
print("Main wells (n=5 per duration):")
print(t_main.round(4).to_string())
neg = (t_main.slope < 0) & (t_main.p < 0.05)
print("negative slope with p < 0.05 in %d of %d readings; p range %.4f to %.4f"
      % (neg.sum(), len(t_main), t_main.p.min(), t_main.p.max()))

In [ ]:
print("Per-well FWHM (mm), 1-mm steps and contiguous ROIs, half-range 5 mm, main wells marked *:")
pw = pd.DataFrame([{"well": w, "pulse_ns": d, "main": w in main[d],
                    "fwhm_1mm": cc.fwhm(meas[w], "1mm", 5.0, "avg")["fwhm"],
                    "fwhm_contiguous": cc.fwhm(meas[w], "contiguous", 5.0, "avg")["fwhm"],
                    "R_fit_contiguous": cc.fwhm(meas[w], "contiguous", 5.0, "avg")["r"]} for d in (300, 600, 900) for w in all7[d]])
print(pw.round(3).to_string())
t_all = fig6d_slopes(all7)
neg7 = (t_all.slope < 0) & (t_all.p < 0.05)
print()
print("All 7 wells at 600 ns: negative slope with p < 0.05 in %d of %d readings; p range %.3f to %.3f; mean FWHM at 600 ns %.2f to %.2f mm"
      % (neg7.sum(), len(t_all), t_all.p.min(), t_all.p.max(), t_all.fwhm600.min(), t_all.fwhm600.max()))
sub_rows = []
for s in subsets:
    t = fig6d_slopes(sets_for(s))
    sub_rows.append({"left out": tuple(sorted(set(six) - set(s))), "readings significant": int(((t.slope < 0) & (t.p < 0.05)).sum()),
                     "p (1mm, 5 mm, individual, linear)": float(t[(t.spacing == "1mm") & (t.half_mm == 5) & (t.pooling == "avg") & (t.points == "individual") & (t.trend == "linear")].p.iloc[0])})
print(pd.DataFrame(sub_rows).round(4).to_string())

In [ ]:
print("Main wells, background from the mean of all 4 footprints (variant):")
rows = []
for spacing, half in itertools.product(cc.SPACINGS, cc.HALF_RANGES):
    f = fw(main, spacing, half, "avg", bg="all")
    xs = np.concatenate([[d] * 5 for d in (300, 600, 900)])
    st = cc.slope_test(xs, np.concatenate([f[d] for d in (300, 600, 900)]))
    rows.append({"spacing": spacing, "half_mm": half, "slope_per_ns": st["slope"], "p": st["p"]})
print(pd.DataFrame(rows).round(5).to_string())
print()
print("FWHM by plate (main wells, contiguous ROIs, 5 mm):")
pw["plate"] = [wl.loc[w, "plate"] for w in pw.well]
print(pw[pw["main"]].pivot_table(index="plate", columns="pulse_ns", values="fwhm_contiguous").round(2))

**Result: blocked (methods underspecified): the readings differ, which the criterion says
blocks the claim.** With the five complete plate rows, mean FWHM falls from 2.11 to 2.13 mm at
300 ns to 1.80 to 1.91 mm at 600 ns and 1.69 to 1.80 mm at 900 ns, and the slope is negative under
all 48 readings. It is significant (p < 0.05) under 28: every reading fitted to the 15 individual
values with contiguous ROIs (p 0.0020 to 0.0037, close to the paper's 0.003), 10 of 12 with 1-mm
steps (p 0.042 to 0.050), and the log trend on the 3 means with 1-mm steps (p 0.012 to 0.016); the
other readings on 3 means give p 0.08 to 0.18. FWHM falls with duration on each plate. With all
7 wells at 600 ns the slope is significant under none of the 48 readings (p 0.83 to 1.00), as the
two P1 row B wells have FWHMs of about 5 and 18 mm; of the 21 choices of 5 of the 7, only the one
that leaves out both of them gives a significant slope. The outcome
therefore depends on leaving out the two P1 row B wells: with either of them kept, the slope is
significant under no reading and the claim would be not reproduced. With distances from the OME
pixel size instead of scaling the footprint centre to 9.05 mm (cell under fig6d-900ns-fwhm), the
linear slope on the 15 values gives p 0.0044 to 0.0051 with contiguous ROIs and 0.040 to 0.045
with 1-mm steps, the same pattern. Contiguous ROIs overlap (a 0.406-mm step along a diagonal near
45 degrees moves an axis-aligned box 0.29 mm along each axis), which does not enter the slope
test on per-well FWHMs.

## fig6d-600-900-differ-from-300

> FWHM for 600-ns and for 900-ns packets is each significantly smaller than for 300-ns packets
> (Bonferroni-corrected t-tests, p = 0.028 for one and p = 0.0072 for the other).

Two-sided Student's t-tests of 600 against 300 ns and 900 against 300 ns, Bonferroni over 2 or 3
comparisons, under every reading of ROI spacing, fit range and pooling.

In [ ]:
def fig6d_pairs(wset):
    rows = []
    for spacing, half, pooling in readings:
        f = fw(wset, spacing, half, pooling)
        for m in (2, 3):
            a = cc.ttest_bonf(f[600], f[300], m)
            b = cc.ttest_bonf(f[900], f[300], m)
            rows.append({"spacing": spacing, "half_mm": half, "pooling": pooling, "m": m,
                         "p600v300": a["p_bonf"], "diff600": a["diff"], "d600": a["cohen_d"],
                         "p900v300": b["p_bonf"], "diff900": b["diff"], "d900": b["cohen_d"]})
    t = pd.DataFrame(rows)
    t["n_significant"] = ((t.p600v300 < 0.05) & (t.diff600 < 0)).astype(int) + ((t.p900v300 < 0.05) & (t.diff900 < 0)).astype(int)
    return t

p_main = fig6d_pairs(main)
print("Main wells:")
print(p_main.round(4).to_string())
print("readings with both significant: %d, one: %d, none: %d of %d"
      % ((p_main.n_significant == 2).sum(), (p_main.n_significant == 1).sum(), (p_main.n_significant == 0).sum(), len(p_main)))
p_all = fig6d_pairs(all7)
print("All 7 at 600 ns: readings with both significant %d, one %d, none %d; 600v300 corrected p %.3f to %.3f"
      % ((p_all.n_significant == 2).sum(), (p_all.n_significant == 1).sum(), (p_all.n_significant == 0).sum(), p_all.p600v300.min(), p_all.p600v300.max()))

**Result: blocked (methods underspecified), as the criterion requires when readings differ, and
because it depends on fig6d-longer-pulses-sharper-peak, which is blocked.** In the main wells, 900
against 300 ns is significant after correction with contiguous ROIs (p 0.0053 to 0.0089, close to
the paper's 0.0072) but not with 1-mm steps (p 0.077 to 0.131), and 600 against 300 ns is
significant under no reading (p 0.058 to 0.092 with contiguous ROIs, 0.44 to 0.66 with 1-mm steps,
against the paper's 0.028). The readings give one significant comparison or none; FWHM is smaller
at both longer durations under every reading (Cohen's d -0.84 to -2.71). With unscaled
distances the Bonferroni p over 2 comparisons is 0.0087 to 0.0096 (900 against 300 ns) and 0.091
to 0.092 (600 against 300 ns) with contiguous ROIs, and 0.067 to 0.076 and 0.31 to 0.32 with 1-mm
steps.

## fig6d-900ns-fwhm

> Packets of 900-ns pulses reduce FWHM to 1.7 +/- 0.08 mm (mean +/- s.e.), 11.1 +/- 0.6% of the
> 15.3-mm gap between electrodes.

Mean of the per-well FWHM over the five 900-ns wells (all five are in the dataset, so this claim
does not depend on the choice of 600-ns wells), and the mean as a percent of 15.3 mm.

In [ ]:
rows = []
for spacing, half, pooling in readings:
    v = np.array([cc.fwhm(meas[w], spacing, half, pooling)["fwhm"] for w in main[900]])
    m, s = cc.mean_se(v)
    rows.append({"spacing": spacing, "half_mm": half, "pooling": pooling, "fwhm_mean": m, "fwhm_se": s,
                 "pct_of_15.3": 100 * m / cc.DIAGONAL_GAP_MM, "pct_se": 100 * s / cc.DIAGONAL_GAP_MM,
                 "fwhm_ok (1.62-1.78)": abs(m - 1.7) <= 0.08 + 1e-12, "pct_ok (10.5-11.7)": abs(100 * m / 15.3 - 11.1) <= 0.6 + 1e-12})
t900 = pd.DataFrame(rows)
print(t900.round(3).to_string())
print("readings with both values within tolerance: %d of %d" % ((t900["fwhm_ok (1.62-1.78)"] & t900["pct_ok (10.5-11.7)"]).sum(), len(t900)))
print("per-well 900-ns FWHM, contiguous, 5 mm:", [round(cc.fwhm(meas[w], "contiguous", 5.0, "avg")["fwhm"], 2) for w in main[900]])

In [ ]:
# variant: distances from the OME pixel size (1.625 um), without scaling the footprint centre to 9.05 mm
meas_u = {w: {(("darkest", sp)): cc.diagonal_profiles(yp[w], meas[w]["geom"], sp, meas[w][("bg", "darkest")], scaled=False)
              for sp in cc.SPACINGS} for d in (300, 600, 900) for w in main[d]}
rows = []
for spacing, half, pooling in readings:
    f = {d: np.array([cc.fwhm(meas_u[w], spacing, half, pooling)["fwhm"] for w in main[d]]) for d in (300, 600, 900)}
    m, s = cc.mean_se(f[900])
    xs = np.repeat([300, 600, 900], 5)
    st = cc.slope_test(xs, np.concatenate([f[d] for d in (300, 600, 900)]))
    rows.append({"spacing": spacing, "half_mm": half, "pooling": pooling, "fwhm900_mean": m, "fwhm900_se": s,
                 "pct_of_15.3": 100 * m / cc.DIAGONAL_GAP_MM,
                 "both_ok": abs(m - 1.7) <= 0.08 + 1e-12 and abs(100 * m / 15.3 - 11.1) <= 0.6 + 1e-12,
                 "slope p (15 values, linear)": st["p"], "slope": st["slope"],
                 "p_bonf 600v300 (2)": cc.ttest_bonf(f[600], f[300], 2)["p_bonf"],
                 "p_bonf 900v300 (2)": cc.ttest_bonf(f[900], f[300], 2)["p_bonf"]})
tu = pd.DataFrame(rows)
print("Unscaled distances, main wells:")
print(tu.round(4).to_string())
print("900-ns readings with both values within tolerance: %d of %d" % (tu.both_ok.sum(), len(tu)))

**Result: blocked (methods underspecified), because it depends on
fig6d-longer-pulses-sharper-peak, which is blocked (docs/protocol.md, Author, step 1.5: a claim
that depends on a blocked claim is blocked for the same reason).** Its own criterion would be met:
with contiguous ROIs the mean 900-ns FWHM is 1.69 to 1.71 mm (s.e. 0.08 to 0.09), 11.0 to 11.1%
of 15.3 mm, within both tolerances under all six contiguous readings. With 1-mm steps it is 1.78 to
1.80 mm, just above the upper bound of 1.78 mm. With distances from the OME pixel size, unscaled,
it is 1.69 to 1.71 mm with contiguous ROIs (within both tolerances in 6 of 12 readings) and 1.81
to 1.83 mm with 1-mm steps.

## Fig. 5 at 0-mm elevation: the 600-ns wells

Fig. 5 compares YP and Pr with the electrodes touching the monolayer (0 mm) or raised. Its 0-mm
condition is the MHz-CANCAN protocol, 600-ns packets with no interval, 2 bursts (x4), CT26 cells
(Results 3.2), the same protocol as the dandiset's 600-ns wells, which also have Pr images on
plates P1 and P2. The Fig. 5 legend gives n=6-7 for YP and n=4-5 for Pr; the dandiset has 7 YP and
5 Pr wells at 600 ns. Nothing in the dandiset links these wells to Fig. 5: its title and
description name only the pulse-duration experiment, the raised-electrode wells of the same series
are absent, and Pr images exist also for three 300-ns and three 900-ns wells. The two Fig. 5 claims
are therefore blocked (metadata insufficient). The analysis below shows what they would give if the
600-ns wells were Fig. 5's 0-mm group. Readings: YP from all 7 wells or the 5 main ones; Pr from all
5 wells, the 4 without P1_20240926_B3, or the 3 in the main rows; 1-mm or contiguous ROIs. Fits are
to the mean profile over wells (two diagonals averaged per well), from -5 to 5 mm (Results 3.2); R2
does not depend on the scaling to 100%.

In [ ]:
pr_meas = {w: cc.measure_well(pr[w]) for w in pr_wells}
print("Pr footprints: diagonals (mm)", {w: tuple(round(v, 2) for v in pr_meas[w]["geom"]["diag_lengths_mm"]) for w in pr_wells})
print("Pr background (darkest footprint):", {w: round(pr_meas[w][("bg", "darkest")], 1) for w in pr_wells})
fig, axes = plt.subplots(1, len(pr_wells), figsize=(3.2 * len(pr_wells), 3.4))
for ax, w in zip(axes, pr_wells):
    im = pr[w]
    ax.imshow(im, cmap="gray", vmin=np.percentile(im[im > 0], 1), vmax=np.percentile(im, 99.5))
    e = pr_meas[w]["geom"]["electrodes"]
    ax.plot(e[:, 1], e[:, 0], "r+", ms=10)
    ax.set_title(w + " Pr", fontsize=8)
    ax.axis("off")
plt.show()

def mean_profile(mdict, ws, spacing):
    x, _ = mdict[ws[0]][("darkest", spacing)]
    return x, np.array([mdict[w][("darkest", spacing)][1].mean(0) for w in ws]).mean(0)

fit5 = []
for spacing in cc.SPACINGS:
    for yname, yset in (("YP 7 wells", all7[600]), ("YP 5 main", main[600])):
        for pname, pset in (("Pr 5 wells", pr_wells), ("Pr 4 wells", [w for w in pr_wells if w != "P1_20240926_B3"]),
                            ("Pr 3 main", [w for w in pr_wells if w in main[600]])):
            xy, my = mean_profile(meas, yset, spacing)
            xp, mp = mean_profile(pr_meas, pset, spacing)
            yl, yg = cc.fit_curve(xy, my, "lorentz", 5.0), cc.fit_curve(xy, my, "gauss", 5.0)
            pl, pg = cc.fit_curve(xp, mp, "lorentz", 5.0), cc.fit_curve(xp, mp, "gauss", 5.0)
            fit5.append({"spacing": spacing, "YP set": yname, "Pr set": pname,
                         "Pr R2 Lor": pl["r2"], "Pr R2 Gau": pg["r2"], "YP R2 Lor": yl["r2"], "YP R2 Gau": yg["r2"],
                         "max fit p": max(pl["p"], pg["p"], yl["p"], yg["p"]),
                         "YP FWHM Lor": yl["fwhm"], "Pr FWHM Gau": pg["fwhm"], "Pr FWHM Lor": pl["fwhm"]})
fit5 = pd.DataFrame(fit5)

In [ ]:
# Pr images: footprint rim contrast (as for YP above), per-well profiles, and fits with an offset
from scipy import optimize

print("minimum rim contrast at the four footprints, Pr and YP images of the same well:")
print(pd.DataFrame({"Pr min rim contrast": {w: min(pr_meas[w]["geom"]["contrast"]) for w in pr_wells},
                    "Pr contrasts": {w: np.round(pr_meas[w]["geom"]["contrast"], 1) for w in pr_wells},
                    "YP min rim contrast": {w: min(meas[w]["geom"]["contrast"]) for w in pr_wells}}).round(1).to_string())
xp1, _ = pr_meas[pr_wells[0]][("darkest", "1mm")]
prt = pd.DataFrame({w: pr_meas[w][("darkest", "1mm")][1].mean(0) for w in pr_wells}, index=xp1).T
print()
print("Background-subtracted Pr per well (two diagonals averaged, 1-mm steps):")
print(prt.round(0).to_string())
print("Pr within +-5 mm (excluding 0) as % of the centre, per well:")
print((100 * prt[[v for v in xp1 if 0 < abs(v) <= 5]].mean(axis=1) / prt[0.0]).round(0).to_string())

def fit_offset(x, y, kind, half=5.0):
    m = np.abs(x) <= half + 1e-9
    xs, ys = x[m], y[m]
    f = (lambda x, a, b, c: cc.lorentz(x, a, b) + c) if kind == "lorentz" else (lambda x, a, s, c: cc.gauss(x, a, s) + c)
    p, _ = optimize.curve_fit(f, xs, ys, p0=(ys.max() - ys.min(), 2.0, ys.min()), maxfev=20000)
    pred = f(xs, *p)
    fw = abs(p[1]) * (1 if kind == "lorentz" else 2.4)
    return 1 - ((ys - pred) ** 2).sum() / ((ys - ys.mean()) ** 2).sum(), fw, p[2]

rows = []
for spacing in cc.SPACINGS:
    for pname, pset in (("Pr 5 wells", pr_wells), ("Pr 3 main", [w for w in pr_wells if w in main[600]])):
        xp, mp = mean_profile(pr_meas, pset, spacing)
        for kind in ("lorentz", "gauss"):
            r2, fwo, c0 = fit_offset(xp, mp, kind)
            rows.append({"spacing": spacing, "Pr set": pname, "kind": kind, "R2 with offset": r2,
                         "R2 no offset": cc.fit_curve(xp, mp, kind, 5.0)["r2"], "FWHM with offset": fwo,
                         "offset as % of centre": 100 * c0 / mp[np.argmin(abs(xp))]})
print()
print("Fits to the mean Pr profile, -5 to 5 mm, with and without a constant offset:")
print(pd.DataFrame(rows).round(3).to_string())

The Pr images are not clean central spots. Per well, Pr within 5 mm of the centre averages 40 to
53% of the centre value in the three main-row wells and about 100% in the two P1 row B wells, so the
mean profile sits on a broad pedestal. With a constant offset added, the Gaussian fits the mean Pr
profile better than the Lorentzian under all 8 readings (R2 0.935 to 0.979 against 0.922 to 0.972),
with offsets of 30 to 48% of the centre and Gaussian FWHMs of 3.2 to 3.6 mm; without it, the
Lorentzian fits better. The Methods' fits have no offset, so whether these Pr data are comparable
with the paper's cannot be settled from the dandiset. In the Pr images the weakest footprint has a
rim contrast of 74 to 76 in P1_20240926_B3 and P2_20240925_A3 against 193 to 416 in the other three,
so the footprints are located less reliably there (two marks in P1_20240926_B3 fall on no visible
footprint in the figure above).

### fig5-yp-lorentzian-pr-gaussian

> The decline of Pr staining (cell killing) from the centre fits a Gaussian better than a
> Lorentzian (R2 0.96 against 0.92), while YP fits a Lorentzian better (0.98 against 0.95),
> fitted from -5 to 5 mm.

In [ ]:
t5 = fit5[["spacing", "YP set", "Pr set", "Pr R2 Lor", "Pr R2 Gau", "YP R2 Lor", "YP R2 Gau", "max fit p"]].copy()
t5["Pr Gaussian better"] = t5["Pr R2 Gau"] > t5["Pr R2 Lor"]
t5["YP Lorentzian better"] = t5["YP R2 Lor"] > t5["YP R2 Gau"]
print(t5.round(4).to_string())
print("expected: Pr Lor 0.92 +/- 0.092, Pr Gau 0.96 +/- 0.096, YP Lor 0.98 +/- 0.098, YP Gau 0.95 +/- 0.095")
for c, e in (("Pr R2 Lor", 0.92), ("Pr R2 Gau", 0.96), ("YP R2 Lor", 0.98), ("YP R2 Gau", 0.95)):
    print("  %s within tolerance in %d of %d readings (range %.3f to %.3f)"
          % (c, (abs(t5[c] - e) <= 0.1 * e).sum(), len(t5), t5[c].min(), t5[c].max()))
print("both orderings hold in %d of %d readings" % ((t5["Pr Gaussian better"] & t5["YP Lorentzian better"]).sum(), len(t5)))

**Result: blocked (metadata insufficient): nothing links the dandiset's 600-ns wells to Fig. 5.**
If they were its 0-mm group, the claim would fail as the criterion states it: the Lorentzian fits
YP better than the Gaussian under all 12 readings (R2 0.93 to 0.99 against 0.75 to 0.97), but it
also fits Pr better under all 12 (R2 0.81 to 0.95 against 0.72 to 0.86), the reverse of the paper,
and the Pr Gaussian R2 is within its tolerance (0.864 to 1.056) under none. With an offset the
ordering for Pr reverses (cell above). The "max fit p" treats overlapping contiguous ROIs as
independent and understates p for those readings.

### fig5-yp-spot-slightly-larger-than-pr

> The central spot of YP staining (electroporation) is only slightly larger than the spot of Pr
> staining (cell killing).

FWHM of the YP Lorentzian and of the Pr Gaussian (2.4 sigma, Methods 2.5) on the mean profiles,
-5 to 5 mm, 0-mm elevation only (no raised-electrode wells in the dandiset).

In [ ]:
t6 = fit5[["spacing", "YP set", "Pr set", "YP FWHM Lor", "Pr FWHM Gau", "Pr FWHM Lor"]].copy()
t6["YP/Pr ratio"] = t6["YP FWHM Lor"] / t6["Pr FWHM Gau"]
t6["YP larger"] = t6["YP FWHM Lor"] > t6["Pr FWHM Gau"]
print(t6.round(3).to_string())
print("YP FWHM larger in %d of %d readings; ratio %.2f to %.2f" % (t6["YP larger"].sum(), len(t6), t6["YP/Pr ratio"].min(), t6["YP/Pr ratio"].max()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, spacing in zip(ax, cc.SPACINGS):
    xy, my = mean_profile(meas, all7[600], spacing)
    xp, mp = mean_profile(pr_meas, pr_wells, spacing)
    a.plot(xy, 100 * my / my[np.argmin(abs(xy))], "go", ms=3, label="YP, 7 wells")
    a.plot(xp, 100 * mp / mp[np.argmin(abs(xp))], "r^", ms=3, label="Pr, 5 wells")
    xx = np.linspace(-7, 7, 300)
    yl = cc.fit_curve(xy, my, "lorentz", 5.0)
    pg = cc.fit_curve(xp, mp, "gauss", 5.0)
    a.plot(xx, 100 * cc.lorentz(xx, yl["a"], yl["fwhm"]) / my[np.argmin(abs(xy))], "g-")
    a.plot(xx, 100 * cc.gauss(xx, pg["a"], pg["fwhm"] / 2.4) / mp[np.argmin(abs(xp))], "r-")
    a.set_title(spacing)
    a.set_xlabel("mm from centre")
    a.set_ylabel("% of centre")
    a.legend()
plt.show()
xy, my = mean_profile(meas, all7[600], "contiguous")
xp, mp = mean_profile(pr_meas, pr_wells, "contiguous")
print("plotted ranges: YP %.0f to %.0f %%, Pr %.0f to %.0f %%" % (100 * my.min() / my[np.argmin(abs(xy))], 100 * my.max() / my[np.argmin(abs(xy))],
                                                                  100 * mp.min() / mp[np.argmin(abs(xp))], 100 * mp.max() / mp[np.argmin(abs(xp))]))

**Result: blocked (metadata insufficient), for the same reason.** If the 600-ns wells were Fig. 5's
0-mm group, the YP spot would be smaller than the Pr spot under all 12 readings: YP Lorentzian
FWHM 1.73 to 2.67 mm against Pr Gaussian FWHM 6.40 to 8.93 mm (ratio 0.20 to 0.41). With an offset
the Pr Gaussian FWHM is 3.2 to 3.6 mm, still larger than the YP spot.

## Parts of blocked claims the dandiset can inform

The claims below are blocked because the dandiset lacks the packet-rate, raised-electrode,
burst-number and thermometry experiments. Two quantities they mention can be measured on the
600-ns wells, which use the 0.2-MHz, 2-burst (x4) protocol: the ratio of centre to near-electrode
YP (fig4b-suppression-near-electrodes, 40-fold) and near-electrode YP as percent of the centre at
0 mm (fig5-raised-electrodes-weaker-centre). They are shown, not judged: the wells are not the
experiments of Fig. 4.

In [ ]:
rows = []
for name, ws in (("5 main", main[600]), ("all 7", all7[600])):
    c = np.array([cc.centre_value(meas[w]) for w in ws])
    for dist in (6.0, 7.0):
        nv = np.array([meas[w][("darkest", "1mm")][1].mean(0)[np.isin(meas[w][("darkest", "1mm")][0], [-dist, dist])].mean() for w in ws])
        rows.append({"600-ns wells": name, "distance_mm": dist, "ratio of means": c.mean() / nv.mean(),
                     "mean of ratios": np.mean(c / nv), "near as % of centre": 100 * nv.mean() / c.mean(),
                     "near YP range": "%.0f to %.0f" % (nv.min(), nv.max())})
print(pd.DataFrame(rows).round(2).to_string())

## Summary

| Claim | Outcome |
|---|---|
| fig6b-peak-900ns-not-significant | reproduced (assumption: Fig. 6 is the five complete plate rows) |
| fig6c-normalized-decline-similar | not reproduced |
| fig6d-longer-pulses-sharper-peak | blocked, methods underspecified (readings differ) |
| fig6d-600-900-differ-from-300 | blocked, methods underspecified (readings differ; depends on the above) |
| fig6d-900ns-fwhm | blocked, methods underspecified (depends on fig6d-longer-pulses-sharper-peak) |
| fig5-yp-lorentzian-pr-gaussian | blocked, metadata insufficient (nothing links the 600-ns wells to Fig. 5) |
| fig5-yp-spot-slightly-larger-than-pr | blocked, metadata insufficient (same reason) |
| fig5-raised-electrodes-weaker-centre | blocked, data missing |
| fig3-cancan-centre-enhancement, fig3-lorentzian-decline | blocked, data missing |
| fig4b-suppression-near-electrodes, fig4b-centre-no-rate-effect, fig4c-fwhm-falls-with-rate | blocked, data missing |
| fig7be-burst-number-centre, fig7-fwhm-burst-number, fig7-sharpest-peak-1-burst-ct26 | blocked, data missing |
| fig2-heating-below-44c | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: ROIs are placed by an algorithm on the footprints it
finds, not by eye in MetaMorph, and are rounded to a 40.6-um grid; distances are scaled so the
footprint centre is 9.05 mm from the centre (unscaled distances give the same outcomes); the ROI spacing, fit range and the wells of the
600-ns group are not given by the paper and are run under every reading; the background ROIs are
in the darkest footprint.

What was found about the dandiset: it holds only the pulse-duration experiment (Fig. 6, and
perhaps the 0-mm wells of Fig. 5, which nothing in it identifies); every well's description gives 9 packets at 0.2 MHz whatever
the pulse duration; it has 7 wells at 600 ns where the paper analysed 5, with nothing to say which;
two of them (P1 row B) show uptake across the whole array, and with them the Fig. 6D trend is
absent; one propidium image is deposited twice
(P2_20240925_B3); plate P3 has no propidium images; pixel size and the electrode geometry agree
with the paper.